# Census ETL 

In [ ]:
from pathlib import Path
import os
import shutil
import sys
sys.path.append("..")
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from helpers.data_prep_and_print import print_df
from census_etl_simple import load_simple, create_gold_demographics

WAREHOUSE_PATH = "/tmp/census-dwh"

## Data Warehouse-Struktur aufbauen

In [ ]:

# delete old dwh (optional)
if Path(WAREHOUSE_PATH).exists():
    print(f"delete old dwh: {WAREHOUSE_PATH}")
    shutil.rmtree(WAREHOUSE_PATH)

# create dwh structure
print(f"create dwh structure in: {WAREHOUSE_PATH}")
Path(WAREHOUSE_PATH).mkdir(parents=True, exist_ok=True)
for name in ["bronze", "silver", "staging", "quarantine"]:
    (Path(WAREHOUSE_PATH) / name).mkdir(exist_ok=True)
    print(f"  ✓ {name}")

## Create spark session

In [ ]:
os.environ.pop("SPARK_REMOTE", None)
#create a SparkSession
spark = (SparkSession
       .builder
       .appName("Census ETL")
       .master("local[*]")
       .config("spark.sql.session.timeZone", "UTC")
       .config("spark.sql.shuffle.partitions", "2")
       .getOrCreate())

## Load Census Data for 2010, 2011, 2012

In [ ]:
years_data = [
    (2010, "../data/census/census_2010.json", "census_2010"),
    (2011, "../data/census/census_2011.json", "census_2011"),
    (2012, "../data/census/census_2012.json", "census_2012"),
]

for year, source_file, batch_id in years_data:
    source_path = Path(source_file).resolve()
    print(f"Lade {year}: {batch_id}")
    result = load_simple(
        spark,
        source=str(source_path),
        warehouse=WAREHOUSE_PATH,
        year=year,
        batch_id=batch_id,
    )
    print(f"  → {result}\n")

## Verify Silver Layer Data

In [ ]:
silver_path = Path(WAREHOUSE_PATH) / "silver"
silver = spark.read.option("recursiveFileLookup", "true").parquet(str(silver_path))

print(f"Silver layer contains: {silver.count()} rows\n")
print("Schema:")
silver.printSchema()
print_df(silver.filter(col("year")==2010), 10)

## Transformation to Gold Layer

In [ ]:
for year, source_file, batch_id in years_data:
    print(f"Processing {year}: {batch_id}")
    result = create_gold_demographics(
        spark,
        warehouse=WAREHOUSE_PATH,
        batch_id=batch_id,
    )
    print(f"  → {result}\n")

## Gold Layer Analysis

In [ ]:
gold_path = Path(WAREHOUSE_PATH) / "gold"
demographics = spark.read.option("recursiveFileLookup", "true").parquet(str(gold_path))

print(f"Gold layer contains: {demographics.count()} rows\n")
print("Schema:")
demographics.printSchema()
print_df(demographics.orderBy("year", "age_from"))

In [ ]:
spark.stop()